In [43]:
import torch
import torch.nn as nn
import math

In [33]:
VOCAB_SIZE = 50257
BLOCK_SIZE = 256
N_EMBD = 128

HEAD_SIZE = 32

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

# Embeddings

In [ ]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        self.block_size = block_size
        self.tok_embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(block_size, n_embd)


    def forward(self, x):
        size = x.size(dim=-1)
        if (size > self.block_size):
            raise ValueError(f"Input size ({size}) exceded max block size ({self.block_size})")

        idx = torch.arange(size).to(device)

        pos_emb = self.pos_embedding(idx)
        tok_emb = self.tok_embedding(x)

        emb = tok_emb + pos_emb

        return emb


# Self Attention

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, head_size, n_embd):
        super().__init__()
        self.head_size = head_size
        self.Q = nn.Linear(n_embd, head_size)
        self.K = nn.Linear(n_embd, head_size)
        self.V = nn.Linear(n_embd, head_size)

    def forward(self, x):
        q = self.Q(x)
        k = self.K(x)
        v = self.V(x)

        attention_score = (q @ k.transpose(-1,-2)) / math.sqrt(self.head_size)

        causal_mask = torch.tril(
            torch.ones(
                (attention_score.size(dim=-1), 
                 attention_score.size(dim=-2)), 
                 dtype=torch.bool
            )
        )

        masked_attention_score = attention_score.masked_fill(~causal_mask, -torch.inf)
        score = torch.softmax(masked_attention_score, dim=-1)

        res = score @ v
        return res

In [79]:
emb = Embedding(VOCAB_SIZE, BLOCK_SIZE, N_EMBD)
x = emb(torch.randint(0, 10, (4, 200)))

att = SelfAttention(HEAD_SIZE, N_EMBD)
x = att(x)

torch.Size([4, 200, 200])
